# Step 3A - Data-driven Target Selection (No Anchors)

本步骤仅依赖 `scCRC_ICB` DEG 数据进行候选靶点优先级计算：

- 不使用预设锚点
- 不硬编码任何特定基因
- 仅依据统计显著性与效应量排序

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve().parent.parent if Path.cwd().name == "example_sccrc_icb_step_by_step" else Path.cwd().resolve()
deg_dir = ROOT / "data" / "scRNA" / "scCRC_ICB" / "deg_tables"
out_dir = ROOT / "results" / "examples" / "sccrc_icb" / "step3"
out_dir.mkdir(parents=True, exist_ok=True)

frames = []
for f in sorted(deg_dir.glob("*.csv")):
    df = pd.read_csv(f)
    required = {"gene", "avg_log2FC", "p_val_adj"}
    if not required.issubset(df.columns):
        continue
    tmp = df[["gene", "avg_log2FC", "p_val_adj"]].copy()
    tmp["source_file"] = f.name
    tmp["gene"] = tmp["gene"].astype(str).str.upper()
    tmp["p_val_adj"] = pd.to_numeric(tmp["p_val_adj"], errors="coerce")
    tmp["avg_log2FC"] = pd.to_numeric(tmp["avg_log2FC"], errors="coerce")
    tmp = tmp.dropna(subset=["gene", "avg_log2FC", "p_val_adj"])
    frames.append(tmp)

if not frames:
    raise RuntimeError("No valid DEG tables found for data-driven ranking.")

all_deg = pd.concat(frames, ignore_index=True)
all_deg["score"] = all_deg["avg_log2FC"].abs() * (-np.log10(all_deg["p_val_adj"].clip(lower=1e-300)))

ranked = (
    all_deg.groupby("gene", as_index=False)
    .agg(
        mean_abs_log2fc=("avg_log2FC", lambda x: float(np.mean(np.abs(x)))),
        median_score=("score", "median"),
        max_score=("score", "max"),
        support_tables=("source_file", "nunique"),
    )
    .sort_values(["median_score", "support_tables", "max_score"], ascending=False)
    .reset_index(drop=True)
)

top_n = 20
top = ranked.head(top_n).copy()
top.to_csv(out_dir / "data_driven_targets_top20.csv", index=False)
print(top.head(10))
print(f"Saved -> {out_dir / 'data_driven_targets_top20.csv'}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_df = top.head(15).iloc[::-1]
ax.barh(plot_df["gene"], plot_df["median_score"])
ax.set_title("Data-driven target ranking (Top 15)")
ax.set_xlabel("median score = |log2FC| * -log10(adj_p)")
ax.set_ylabel("Gene")
plt.tight_layout()
plt.show()